# Monte Carlo Tree Search (MCTS) - A Simple Introduction


This notebook provides a simple, from-scratch implementation of Monte Carlo Tree Search (MCTS), the algorithm that powered AlphaGo and many modern game-playing AIs.

## What is MCTS?

Monte Carlo Tree Search is a search algorithm for decision-making in games and other domains. Instead of exhaustively searching all possible moves (like minimax), MCTS intelligently samples the game tree by:

1. **Focusing on promising moves** - Explores paths that have performed well
2. **Balancing exploration vs exploitation** - Tries new moves while favoring good ones
3. **Using random simulations** - Plays out games randomly to estimate move quality

### The Four Phases

MCTS repeatedly performs four phases:

1. **Selection** - Starting from root, select best child nodes using UCB1 formula
2. **Expansion** - Add a new child node to the tree
3. **Simulation** - Play out a random game from the new node
4. **Backpropagation** - Update statistics back up the tree

```
    Root (visits: 100)
    /    |    \
   A     B     C     <-- Selection: Pick best path
  / \    |
 D   E   F
     |   |
     ?   NEW  <-- Expansion: Add new node
         |
         ... <-- Simulation: Play randomly
         WIN/LOSS <-- Backpropagation: Update stats
```

## Implementation: Tic-Tac-Toe

We'll implement MCTS for Tic-Tac-Toe to keep things simple and visual.

In [ ]:
import numpy as np
import math
import random
from collections import defaultdict
from typing import Optional, List, Tuple
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
import matplotlib.patches as mpatches

### Tic-Tac-Toe Game Implementation

First, we need a simple game environment. We'll represent:
- Empty cells as 0
- Player X as 1
- Player O as -1

In [ ]:
class TicTacToe:
    """Simple Tic-Tac-Toe game."""
    
    def __init__(self):
        self.board = np.zeros((3, 3), dtype=int)
        self.current_player = 1  # 1 for X, -1 for O
    
    def clone(self):
        """Create a copy of the current game state."""
        game = TicTacToe()
        game.board = self.board.copy()
        game.current_player = self.current_player
        return game
    
    def get_legal_moves(self) -> List[Tuple[int, int]]:
        """Return list of (row, col) for empty cells."""
        return [(r, c) for r in range(3) for c in range(3) if self.board[r, c] == 0]
    
    def make_move(self, move: Tuple[int, int]):
        """Play a move and switch players."""
        r, c = move
        self.board[r, c] = self.current_player
        self.current_player = -self.current_player
    
    def check_winner(self) -> Optional[int]:
        """Check if someone won. Returns 1 (X wins), -1 (O wins), 0 (draw), or None (ongoing)."""
        # Check rows and columns
        for i in range(3):
            if abs(self.board[i, :].sum()) == 3:
                return self.board[i, 0]
            if abs(self.board[:, i].sum()) == 3:
                return self.board[0, i]
        
        # Check diagonals
        if abs(self.board.trace()) == 3:
            return self.board[1, 1]
        if abs(np.fliplr(self.board).trace()) == 3:
            return self.board[1, 1]
        
        # Check for draw
        if len(self.get_legal_moves()) == 0:
            return 0
        
        return None  # Game ongoing
    
    def is_terminal(self) -> bool:
        """Check if game is over."""
        return self.check_winner() is not None
    
    def display(self):
        """Pretty print the board."""
        symbols = {0: '.', 1: 'X', -1: 'O'}
        for row in self.board:
            print(' '.join(symbols[cell] for cell in row))
        print()

### Test the Game

Let's verify our game works correctly:

In [ ]:
# Quick test
game = TicTacToe()
game.make_move((0, 0))  # X
game.make_move((1, 0))  # O
game.make_move((0, 1))  # X
game.make_move((1, 1))  # O
game.make_move((0, 2))  # X wins!

game.display()
print(f"Winner: {game.check_winner()}")
print(f"Is terminal: {game.is_terminal()}")

### MCTS Node

Each node in the search tree represents a game state and tracks:
- **visits** (N): How many times we've visited this node
- **wins** (W): Total reward accumulated from simulations
- **children**: Child nodes for each possible move

In [ ]:
class MCTSNode:
    """A node in the MCTS tree."""
    
    def __init__(self, game_state: TicTacToe, parent=None, move=None):
        self.game_state = game_state
        self.parent = parent
        self.move = move  # The move that led to this state
        
        self.visits = 0
        self.wins = 0  # From perspective of player who just moved
        
        self.children = {}  # move -> MCTSNode
        self.untried_moves = game_state.get_legal_moves()
    
    def is_fully_expanded(self) -> bool:
        """Check if all legal moves have been tried."""
        return len(self.untried_moves) == 0
    
    def best_child(self, c_param: float = 1.41) -> 'MCTSNode':
        """Select best child using UCB1 formula.
        
        UCB1 = (wins / visits) + c * sqrt(ln(parent_visits) / visits)
                 ^exploitation      ^exploration
        """
        choices_weights = [
            (child.wins / child.visits) + 
            c_param * math.sqrt(math.log(self.visits) / child.visits)
            for child in self.children.values()
        ]
        return list(self.children.values())[np.argmax(choices_weights)]
    
    def expand(self) -> 'MCTSNode':
        """Expand tree by adding a new child node."""
        move = self.untried_moves.pop()
        next_state = self.game_state.clone()
        next_state.make_move(move)
        child = MCTSNode(next_state, parent=self, move=move)
        self.children[move] = child
        return child
    
    def update(self, result: float):
        """Update node statistics after simulation."""
        self.visits += 1
        self.wins += result

### MCTS Algorithm

Now we implement the main MCTS algorithm with the four phases:

**UCB1 Formula**: The key to MCTS is balancing exploration and exploitation:

$$\text{UCB1} = \frac{w_i}{n_i} + c \sqrt{\frac{\ln N}{n_i}}$$

Where:
- $w_i$ = wins for this node
- $n_i$ = visits to this node
- $N$ = parent's visits
- $c$ = exploration constant (usually √2)

In [ ]:
class MCTS:
    """Monte Carlo Tree Search algorithm."""
    
    def __init__(self, iterations: int = 1000):
        self.iterations = iterations
    
    def search(self, initial_state: TicTacToe) -> Tuple[int, int]:
        """Run MCTS and return the best move."""
        root = MCTSNode(initial_state)
        
        # Run MCTS iterations
        for _ in range(self.iterations):
            node = root
            state = initial_state.clone()
            
            # 1. SELECTION: Traverse tree using UCB1
            while not state.is_terminal() and node.is_fully_expanded():
                node = node.best_child()
                state.make_move(node.move)
            
            # 2. EXPANSION: Add new node if not terminal
            if not state.is_terminal() and not node.is_fully_expanded():
                node = node.expand()
                state.make_move(node.move)
            
            # 3. SIMULATION: Play out randomly
            sim_state = state.clone()
            while not sim_state.is_terminal():
                random_move = random.choice(sim_state.get_legal_moves())
                sim_state.make_move(random_move)
            
            # Get result from perspective of initial player
            result = sim_state.check_winner()
            if result == initial_state.current_player:
                reward = 1  # Win
            elif result == -initial_state.current_player:
                reward = 0  # Loss
            else:
                reward = 0.5  # Draw
            
            # 4. BACKPROPAGATION: Update all nodes in path
            while node is not None:
                node.update(reward)
                reward = 1 - reward  # Flip reward for alternating players
                node = node.parent
        
        # Return most visited move (most robust)
        best_move = max(root.children.items(), 
                       key=lambda x: x[1].visits)[0]
        
        self.root = root  # Save for visualization
        return best_move

## Testing MCTS

Let's see MCTS in action! We'll have it play against random moves.

In [ ]:
def play_game(mcts_iterations: int = 1000, mcts_player: int = 1, verbose: bool = True):
    """Play a game: MCTS vs Random opponent."""
    game = TicTacToe()
    mcts = MCTS(iterations=mcts_iterations)
    
    if verbose:
        print(f"MCTS (player {mcts_player}) vs Random (player {-mcts_player})\n")
    
    while not game.is_terminal():
        if game.current_player == mcts_player:
            # MCTS move
            move = mcts.search(game)
            if verbose:
                print(f"MCTS plays {move}")
        else:
            # Random move
            move = random.choice(game.get_legal_moves())
            if verbose:
                print(f"Random plays {move}")
        
        game.make_move(move)
        if verbose:
            game.display()
    
    winner = game.check_winner()
    if verbose:
        if winner == mcts_player:
            print("MCTS WINS!")
        elif winner == -mcts_player:
            print("Random wins!")
        else:
            print("Draw!")
    
    return winner

# Play one game
play_game(mcts_iterations=1000, mcts_player=1, verbose=True)

### Evaluate MCTS Performance

Let's run 100 games to see how well MCTS performs:

In [ ]:
def evaluate_mcts(num_games: int = 100, mcts_iterations: int = 1000):
    """Evaluate MCTS against random opponent."""
    results = {'mcts_wins': 0, 'random_wins': 0, 'draws': 0}
    
    print(f"Running {num_games} games (MCTS with {mcts_iterations} iterations)...\n")
    
    for i in range(num_games):
        # Alternate who goes first
        mcts_player = 1 if i % 2 == 0 else -1
        winner = play_game(mcts_iterations, mcts_player, verbose=False)
        
        if winner == mcts_player:
            results['mcts_wins'] += 1
        elif winner == -mcts_player:
            results['random_wins'] += 1
        else:
            results['draws'] += 1
    
    print(f"Results after {num_games} games:")
    print(f"  MCTS wins:   {results['mcts_wins']} ({results['mcts_wins']/num_games*100:.1f}%)")
    print(f"  Random wins: {results['random_wins']} ({results['random_wins']/num_games*100:.1f}%)")
    print(f"  Draws:       {results['draws']} ({results['draws']/num_games*100:.1f}%)")
    
    return results

results = evaluate_mcts(num_games=100, mcts_iterations=1000)

### Impact of Iterations

More iterations = better play (but slower). Let's test different iteration counts:

In [ ]:
iteration_counts = [10, 50, 100, 500, 1000, 2000]
win_rates = []

for iterations in iteration_counts:
    results = evaluate_mcts(num_games=50, mcts_iterations=iterations)
    win_rate = results['mcts_wins'] / 50 * 100
    win_rates.append(win_rate)
    print()

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(iteration_counts, win_rates, 'o-', linewidth=2, markersize=8)
plt.xlabel('MCTS Iterations', fontsize=12)
plt.ylabel('Win Rate (%)', fontsize=12)
plt.title('MCTS Performance vs Number of Iterations', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.ylim(0, 100)
plt.axhline(y=50, color='r', linestyle='--', alpha=0.5, label='Random baseline')
plt.legend()
plt.tight_layout()
plt.show()

## Visualizing the Search Tree

Let's peek inside MCTS to see which moves it's considering:

In [ ]:
def visualize_root_statistics(mcts: MCTS):
    """Visualize the root node's children statistics."""
    root = mcts.root
    
    if not root.children:
        print("No children to visualize")
        return
    
    moves = []
    visits = []
    win_rates = []
    
    for move, child in sorted(root.children.items()):
        moves.append(f"{move}")
        visits.append(child.visits)
        win_rates.append(child.wins / child.visits * 100 if child.visits > 0 else 0)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Visits bar chart
    bars1 = ax1.bar(moves, visits, color='steelblue', alpha=0.8)
    ax1.set_xlabel('Move (row, col)', fontsize=11)
    ax1.set_ylabel('Number of Visits', fontsize=11)
    ax1.set_title('MCTS Visit Counts by Move', fontsize=13, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=9)
    
    # Win rates bar chart
    bars2 = ax2.bar(moves, win_rates, color='coral', alpha=0.8)
    ax2.set_xlabel('Move (row, col)', fontsize=11)
    ax2.set_ylabel('Win Rate (%)', fontsize=11)
    ax2.set_title('MCTS Win Rates by Move', fontsize=13, fontweight='bold')
    ax2.set_ylim(0, 100)
    ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax2.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, wr in zip(bars2, win_rates):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{wr:.1f}%',
                ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Print best move
    best_move = max(root.children.items(), key=lambda x: x[1].visits)[0]
    print(f"\nMost visited move (MCTS choice): {best_move}")

# Run MCTS on a position and visualize
game = TicTacToe()
game.display()

mcts = MCTS(iterations=1000)
best_move = mcts.search(game)

visualize_root_statistics(mcts)

## Key Insights

### Why MCTS Works

1. **Adaptive**: Focuses computational effort on promising moves
2. **Anytime**: Can return best move found so far if interrupted
3. **Asymmetric**: Explores interesting branches deeply, ignores bad ones
4. **No evaluation function needed**: Uses random playouts instead

### The UCB1 Formula

The magic of MCTS lies in UCB1:

$$\frac{w_i}{n_i} + c\sqrt{\frac{\ln N}{n_i}}$$

- **First term**: Exploitation - prefer moves with high win rates
- **Second term**: Exploration - try moves we haven't explored much
- **$c$ parameter**: Controls exploration (higher = more exploration)

### Limitations

- **Computationally intensive**: Needs many iterations for complex games
- **Weak in tactical games**: Random playouts don't capture tactics well
- **Cold start problem**: Initial random playouts can be misleading

### Extensions

Modern improvements:
- **Neural MCTS** (AlphaGo, AlphaZero): Replace random playouts with neural network evaluation
- **RAVE (Rapid Action Value Estimation)**: Share statistics across similar positions
- **Progressive widening**: Gradually expand move choices
- **Transposition tables**: Reuse subtree results

## Experiments to Try

1. **Change exploration constant**: Try different values of `c_param` in `best_child()`. How does it affect play?

2. **MCTS vs MCTS**: Have two MCTS agents play each other with different iteration counts

3. **Different games**: Implement MCTS for Connect 4 or another game

4. **Smarter simulations**: Replace random playouts with simple heuristics

5. **Tree reuse**: Keep the tree between moves instead of rebuilding

6. **Visualization**: Draw the full search tree (careful, it gets big!)

## Summary

You've learned:
- ✓ The four phases of MCTS: Selection, Expansion, Simulation, Backpropagation
- ✓ The UCB1 formula for balancing exploration vs exploitation
- ✓ How to implement MCTS from scratch
- ✓ How iteration count affects playing strength
- ✓ How to visualize MCTS decision-making

MCTS is a powerful, general algorithm that has revolutionized game AI. Combined with neural networks (as in AlphaGo), it can master even the most complex games!